## CREATE DATA MASKING FUNCTION

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS raw;
USE CATALOG databricksdemo;
USE SCHEMA raw;
CREATE VOLUME IF NOT EXISTS data;

In [0]:
%sql
DROP FUNCTION IF EXISTS dynamic_mask;

CREATE FUNCTION dynamic_mask(email STRING)
  RETURN CASE
    WHEN is_account_group_member("developers") THEN email
    ELSE "*******"
  END;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ustomers AS
SELECT
  *
FROM
  read_files(
    "/Volumes/databricksdemo/raw/data",
    format => "csv",
    inferSchema => true,
    header => true
  )

In [0]:
%sql
SELECT * EXCEPT (_rescued_data, location, name)
FROM customers
LIMIT 10;

In [0]:
%sql
ALTER TABLE customers ALTER COLUMN email SET MASK dynamic_mask;

In [0]:
%sql
SELECT *
FROM customers;

## ROW LEVEL SECURITY

In [0]:
%sql
DROP TABLE mapping_table;

CREATE TABLE mapping_table (
  user_id STRING,
  region STRING
)

In [0]:
%sql
INSERT INTO mapping_table
VALUES ('so2439a@american.edu', 'east'), ('john.smith@american.edu', 'west'), ('jane.doe@american.edu', 'north'), ('jimmy.smith@american.edu', 'south');

In [0]:
%sql
SELECT EXISTS
(
SELECT *
FROM mapping_table
WHERE user_id = current_user() AND region = 'west'
)

## CREATE RLS FUNCTION


In [0]:
%sql
DROP TABLE stores;

CREATE TABLE stores AS
SELECT
  *
FROM
  read_files(
    "/Volumes/databricksdemo/raw/data",
    format => "csv",
    inferSchema => true,
    header => true
  )

In [0]:
%sql
SELECT * FROM stores;

In [0]:
%sql
CREATE OR REPLACE FUNCTION rls_funct(p_region STRING)
RETURNS BOOLEAN
RETURN EXISTS
(
SELECT *
FROM mapping_table
WHERE user_id = current_user() AND region = p_region
)
    

In [0]:
%sql
ALTER TABLE stores SET ROW FILTER rls_funct ON (region);

In [0]:
%sql
SELECT * FROM stores;

In [0]:
%sql
DROP FUNCTION rls_funct;

In [0]:
%sql
UPDATE stores
SET region = lower(region);

In [0]:
%sql
SELECT * FROM stores;